# Polynomial Regression — from scratch

Polynomial regression is **linear regression on non-linear features**. For one input $x$ and degree $p$:

$$\hat{y} = w_0 + w_1x + w_2x^2 + \dots + w_px^p$$

The model is still linear in the weights, so we can expand the features and reuse least squares.
For several inputs, the expansion contains every monomial up to degree $p$
(e.g. for $x_1, x_2$ and $p=2$: $1, x_1, x_2, x_1^2, x_1x_2, x_2^2$).

High degrees overfit, so we also add an optional **L2 (ridge) penalty**:

$$\mathbf{w} = (X^\top X + \lambda I)^{-1}X^\top y$$

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

from itertools import combinations_with_replacement

In [ ]:
def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

## Implementation

In [ ]:
def polynomial_features(X, degree, include_bias=True):
    """All monomials of the columns of X up to `degree`."""
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    n, d = X.shape
    columns = [np.ones(n)] if include_bias else []
    for deg in range(1, degree + 1):
        for combo in combinations_with_replacement(range(d), deg):
            columns.append(np.prod(X[:, combo], axis=1))
    return np.column_stack(columns)


class PolynomialRegression:
    """Polynomial feature expansion + (optionally ridge-regularised) least squares."""

    def __init__(self, degree=2, alpha=0.0):
        self.degree = degree
        self.alpha = alpha            # ridge strength (0 = ordinary least squares)

    def _transform(self, X):
        P = polynomial_features(X, self.degree, include_bias=False)
        return (P - self.mean_) / self.std_

    def fit(self, X, y):
        P = polynomial_features(X, self.degree, include_bias=False)
        # standardise the expanded features: x^10 is huge compared to x
        self.mean_, self.std_ = P.mean(axis=0), P.std(axis=0) + 1e-12
        Z = (P - self.mean_) / self.std_
        Zb = np.c_[np.ones(len(Z)), Z]
        reg = self.alpha * np.eye(Zb.shape[1])
        reg[0, 0] = 0                 # never penalise the intercept
        theta = np.linalg.pinv(Zb.T @ Zb + reg) @ Zb.T @ np.asarray(y, dtype=float)
        self.b, self.w = theta[0], theta[1:]
        return self

    def predict(self, X):
        return self._transform(X) @ self.w + self.b


def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

In [ ]:
print(polynomial_features(np.array([[2, 3]]), degree=2))   # [1, x1, x2, x1^2, x1*x2, x2^2]

## Data: a noisy cubic

$y = 0.5x^3 - 2x^2 + x + 2 + \varepsilon$

In [ ]:
n = 60
x = np.sort(np.random.uniform(-3, 3, n))
y = 0.5 * x**3 - 2 * x**2 + x + 2 + np.random.normal(0, 2, n)
X = x.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
x_plot = np.linspace(-3, 3, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, deg in zip(axes, [1, 3, 15]):
    model = PolynomialRegression(degree=deg).fit(X_train, y_train)
    ax.scatter(X_train, y_train, label="train", alpha=0.7)
    ax.scatter(X_test, y_test, label="test", marker="x", color="k")
    ax.plot(x_plot, model.predict(x_plot), "r", lw=2)
    tr = mean_squared_error(y_train, model.predict(X_train))
    te = mean_squared_error(y_test, model.predict(X_test))
    ax.set_title(f"degree {deg}\ntrain MSE={tr:.2f}  test MSE={te:.2f}")
    ax.set_ylim(y.min() - 5, y.max() + 5)
    ax.legend()
plt.tight_layout(); plt.show()

## Bias–variance trade-off

Degree 1 **underfits** (high bias). Degree 15 **overfits** (high variance): low training error but
poor test error. We sweep the degree to see the classic U-shaped test curve.

In [ ]:
degrees = range(1, 16)
train_err, test_err = [], []
for d in degrees:
    m = PolynomialRegression(degree=d).fit(X_train, y_train)
    train_err.append(mean_squared_error(y_train, m.predict(X_train)))
    test_err.append(mean_squared_error(y_test, m.predict(X_test)))

plt.plot(degrees, train_err, "o-", label="train MSE")
plt.plot(degrees, test_err, "s-", label="test MSE")
plt.yscale("log"); plt.xlabel("polynomial degree"); plt.ylabel("MSE")
plt.title("Model complexity vs error"); plt.legend(); plt.show()
print("Best degree on test set:", list(degrees)[int(np.argmin(test_err))])

## Ridge regularisation tames a high-degree model

Keeping degree 15 but adding an L2 penalty shrinks the weights and smooths the curve.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, alpha in zip(axes, [0, 0.1, 10]):
    model = PolynomialRegression(degree=15, alpha=alpha).fit(X_train, y_train)
    ax.scatter(X_train, y_train, alpha=0.7)
    ax.scatter(X_test, y_test, marker="x", color="k")
    ax.plot(x_plot, model.predict(x_plot), "r", lw=2)
    te = mean_squared_error(y_test, model.predict(X_test))
    ax.set_title(f"degree 15, alpha={alpha}\ntest MSE={te:.2f}, ||w||={np.linalg.norm(model.w):.1f}")
    ax.set_ylim(y.min() - 5, y.max() + 5)
plt.tight_layout(); plt.show()

## Polynomial regression with two features

Fit $z = x_1^2 + x_1x_2 - x_2^2$ with a degree-2 model in two variables.

In [ ]:
X2 = np.random.uniform(-2, 2, size=(200, 2))
z = X2[:, 0]**2 + X2[:, 0] * X2[:, 1] - X2[:, 1]**2 + np.random.normal(0, 0.1, 200)

model2 = PolynomialRegression(degree=2).fit(X2, z)
print("Train MSE:", round(mean_squared_error(z, model2.predict(X2)), 4))

g = np.linspace(-2, 2, 50)
G1, G2 = np.meshgrid(g, g)
Z = model2.predict(np.c_[G1.ravel(), G2.ravel()]).reshape(G1.shape)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(G1, G2, Z, alpha=0.5, cmap="viridis")
ax.scatter(X2[:, 0], X2[:, 1], z, c="r", s=8)
ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.set_zlabel("z")
ax.set_title("Degree-2 surface fit"); plt.show()

## Summary
* Polynomial regression = feature expansion + linear regression.
* Degree controls model capacity; choose it with a validation/test set.
* Standardising expanded features keeps the solve numerically stable; ridge reduces overfitting.